# Views

In [1]:
import os
from dotenv import load_dotenv

import pandas as pd
import sqlalchemy

In [2]:
load_dotenv()

db_host = os.environ.get("db_host")
db_user = os.environ.get("db_user")
db_password = os.environ.get("db_password")

In [6]:
engine = sqlalchemy.create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/sql_invoicing")

In [7]:
pd.read_sql("SHOW TABLES", con= engine)

,Tables_in_sql_invoicing
0,clients
1,invoices
2,payment_methods
3,payments


## Creating Views
a view behaves like a table, but it doesn't store data 

In [14]:
query = sqlalchemy.text("""
create view sales_by_client as 
select 
	c.client_id,
	c.name,
    sum(invoice_total) as total_sales
from clients c
join invoices i using (client_id)
group by c.client_id
""")

with engine.begin() as conn:
    conn.execute(query)

EXERCISE:

In [12]:
query = sqlalchemy.text("""
create view clients_balance as
select 
	client_id,
    c.name,
    sum(invoice_total - payment_total) as balance
from  invoices i
join clients c using (client_id)
group by client_id
""")

with engine.begin() as conn:
    conn.execute(query)

## Altering or Dropping Views

In [15]:
query = sqlalchemy.text("""
DROP VIEW sales_by_client
""")

with engine.begin() as conn:
    conn.execute(query)

In [16]:
query = sqlalchemy.text("""
create or replace view clients_balance as
select 
	client_id,
    c.name,
    sum(invoice_total - payment_total) as balance
from  invoices i
join clients c using (client_id)
group by client_id
""")



with engine.begin() as conn:
    conn.execute(query)

## WITH CHECK OPTION
The `WITH CHECK OPTION` ensures that any data inserted or updated through a view must satisfy the view’s underlying filter criteria.

It is primarily used to maintain data integrity and prevent unauthorized or inconsistent records from being added to the view.

IMPORTANT: to use this clause your view must be UPDATABLE!!

In [19]:
query = sqlalchemy.text("""
create or replace view clients_balance as
select 
	client_id,
    invoice_total - payment_total as balance
from  invoices 
with check option
""")



with engine.begin() as conn:
    conn.execute(query)

#### Views are crucial for three main reasons:

`Query Simplification`: They encapsulate complex JOIN logic, allowing developers to query pre-defined, clean data sets instead of rewriting complex SQL every time.

`Abstraction (Independence)`: They act as an interface between the database schema and applications. If the underlying table structure changes, we only need to update the view, preventing breaking changes in the application layer.

`Security`: They allow us to restrict data access by exposing only the specific columns or rows that a user is authorized to see, ensuring sensitive information remains protected